In [1]:
model_name = "vit-ragdoll"


import iree
import iree.compiler
import iree.runtime
import torch
import numpy as np
from torch import nn
from torchvision import models

import warnings
warnings.filterwarnings("ignore")

from timeit import timeit as ti
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

from ragdoll.compiler import *
from ragdoll.benchmark import *
import ragdoll

def get_dataframe(backward, item):
    return pd.concat([
        pd.DataFrame({
            "time": backward,
            "pass": "Backward",
            "item": item
        }, index=[0]),
    ])

def timeit(stmt, n=1):
    return ti(stmt, globals=globals(), number=n) * 1000 / n



BENCHMARK_REPEAT=33
df = pd.DataFrame()

def load_executable(fb_file):
    config = iree.runtime.system_api.Config("cuda")
    vmi = iree.runtime.VmInstance()
    # replace compile with args of fatbin type
    # fb_file = ragdoll.compile(mlir, "gpu", "input", "codegen", benchmark=True)
    with open(fb_file, 'rb') as f:
        binary_data = f.read()
    vmm = iree.runtime.VmModule.from_flatbuffer(vmi, binary_data)
    vmo = iree.runtime.load_vm_module(vmm, config)
    return vmo

In [2]:
import gc
model_file_base = "densenet121.mlir"
strategy = "heuristic"
for bs in [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192, 16384]:
    model_file = model_file_base + ".bs{}".format(bs)
    source_file = model_file + ".{}".format(strategy)
    target_file = source_file + ".vmfb"
    #"""
    # gen model with specified batch-size
    !ragdoll-opt {model_file_base} --ragdoll-autodiff-prepare-batch-size=batchsize={bs} > {model_file}
    
    !ragdoll-opt {model_file}  \
    --canonicalize \
    --enable-cse-in-legalizer \
    --symbol-dce \
    --ragdoll-autodiff-vjp-public-functions='strategy=heuristic' \
    --ragdoll-autodiff-vjp \
    --inline \
    --ragdoll-autodiff-inline-function-call \
    --ragdoll-initialisation \
    --eliminate-empty-tensors \
    --ragdoll-legalise-to-iree-compatibility \
    --ragdoll-raise-linalg-to-tosa \
    --ragdoll-forward-func-removal \
    --canonicalize \
    --cse > {source_file}
    
    !iree-compile {source_file} \
    -o {target_file} \
    --iree-hal-target-backends=cuda \
    --iree-hal-benchmark-dispatch-repeat-count={BENCHMARK_REPEAT} \
    --iree-hal-cuda-llvm-target-arch=sm_70
    #"""
    
    ragdoll_binary = load_executable(target_file)

    image = torch.randn(bs, 3, 224, 224)
    image_np = image.detach().cpu().numpy()
    image_t = torch.randn(bs, 224, 224, 3)
    image_np_t = image_t.detach().cpu().numpy()
    model = models.alexnet().train(False)
    model.load_state_dict({k: torch.ones_like(v) * 0.23421 for k, v in model.state_dict().items()})
    output = model(image)
    grad = torch.randn_like(output)
    grad_np = grad.cpu().numpy()

    try:
        #f1 = timeit("ragdoll_binary.forward(image_np_t)") / BENCHMARK_REPEAT
        #print('ragdoll-opt1-gpu-forward in timeit: ', f1)
        b1 = timeit("ragdoll_binary.dforward(grad_np)") / BENCHMARK_REPEAT
        print('ragdoll-opt1-gpu-forward in timeit: ', b1)
        df = pd.concat([df, get_dataframe(b1, "Ragdoll-Autodiff at batch-size = {}".format(bs))])
        #df = pd.concat([df, get_dataframe(f1, b1, "Ragdoll-Autodiff at batch-size = {}".format(bs))])
    except Exception as e:
        print(f"处理模型时出错，批量大小 {bs}: {e}")
    print(df)

densenet121.mlir.bs1.heuristic:3195:13: error: expected 7 offset values, got 5
    %2445 = linalg.generic {indexing_maps = [#map3, #map1, #map2, #map3], iterator_types = ["parallel", "reduction", "reduction", "parallel", "reduction", "reduction"]} ins(%238, %cst_18, %2444 : tensor<1x56x56x128xf32>, tensor<2x2xf32>, tensor<1x28x28x128xf32>) outs(%cst_4 : tensor<1x56x56x128xf32>) {
            ^
densenet121.mlir.bs1.heuristic:65:3: note: called from
  func.func @dforward(%arg0: tensor<1x1000xf32>, %arg1: tensor<1x128x1x1xf32>, %arg2: tensor<1x992x7x7xf32>, %arg3: tensor<1x992x7x7xf32>, %arg4: tensor<1x992x1x1xf32>, %arg5: tensor<1x128x7x7xf32>, %arg6: tensor<1x128x7x7xf32>, %arg7: tensor<1x128x1x1xf32>, %arg8: tensor<1x960x7x7xf32>, %arg9: tensor<1x960x7x7xf32>, %arg10: tensor<1x960x1x1xf32>, %arg11: tensor<1x128x7x7xf32>, %arg12: tensor<1x128x7x7xf32>, %arg13: tensor<1x128x1x1xf32>, %arg14: tensor<1x928x7x7xf32>, %arg15: tensor<1x928x7x7xf32>, %arg16: tensor<1x928x1x1xf32>, %arg17: tens

FileNotFoundError: [Errno 2] No such file or directory: 'densenet121.mlir.bs1.heuristic.vmfb'

In [ ]:
df.style.hide(axis="index")
plt.rcParams["figure.dpi"] = 300

sns.barplot(df, x="pass", y="time", hue="item")
plt.xlabel("Pass")
plt.ylabel("Time Normalized (ms)")
plt.legend().set_title("Item")
plt.title(model_name)

plt.savefig(f"{model_name}-time.png")